# pandas：把資料整理成 ML 可用的表格

已學過 vibe coding，本篇只保留讀資料、檢查、篩選、處理缺失值與準備特徵。
不逐一練習 Series 宣告、索引變體、算術 API 或 apply 家族。
套件安裝請見 [使用說明](./README.md)。

## 1. 讀取與檢查

DataFrame 是表格：一列（row）是一筆樣本，一欄（column）是一個欄位；單一欄是 Series。
使用隨附的 **12 筆模擬氣象資料**，不用下載外部檔案。
從本 notebook 所在的 `simplified` 目錄執行。

In [ ]:
import pandas as pd

df = pd.read_csv('./data/weather_sample.csv')
display(df.head())
df.info()
display(df.describe())

**檢查重點：**資料有幾筆？數字欄位是否讀成數字？有沒有缺失或不合理的範圍？
`describe()` 的 count 不含缺失值，不能直接當作總樣本數。

## 2. 選欄位與篩樣本

`df[欄位列表]` 選特徵；`df.loc[條件, 欄位]` 選符合條件的資料。
`iloc` 按位置取值，例如 `df.iloc[:3]` 是前三筆。

In [ ]:
features = df[['Temperature', 'Humidity']]
warm_days = df.loc[df['Temperature'] > 22, ['Day', 'Temperature']]
print("特徵表 shape：", features.shape)
display(warm_days)

## 3. 缺失值：先觀察，再決定策略

`isna().sum()` 檢查各欄缺失筆數；`fillna()` 可填補，`dropna()` 可刪除缺失樣本。
本例用中位數填補氣溫，只示範操作；填補或刪除都可能改變資料分布。
**實際 ML 要先切分資料，再由訓練集學習填補值**，驗證／測試集使用同一個值。

In [ ]:
print(df.isna().sum())
clean = df.copy()
clean['Temperature'] = clean['Temperature'].fillna(df['Temperature'].median())
print("填補後剩餘缺失數：", clean['Temperature'].isna().sum())

### 小練習 1：用訓練集處理缺失值

使用原始 `df` 的前 6 筆作為訓練集，其餘作為驗證集。
請只用訓練集的氣溫中位數填補兩組資料中的氣溫缺失值，
印出填補值、兩組填補前後的缺失筆數，以及驗證集第 7 天的填補結果。
操作時保留原始 `df` 不變。

<details>
<summary>參考答案</summary>

```python
ex_train = df.iloc[:6].copy()
ex_valid = df.iloc[6:].copy()
train_median = ex_train['Temperature'].median()
print('填補值：', train_median)
print('填補前：', ex_train['Temperature'].isna().sum(),
      ex_valid['Temperature'].isna().sum())
ex_train['Temperature'] = ex_train['Temperature'].fillna(train_median)
ex_valid['Temperature'] = ex_valid['Temperature'].fillna(train_median)
print('填補後：', ex_train['Temperature'].isna().sum(),
      ex_valid['Temperature'].isna().sum())
print(ex_valid.loc[ex_valid['Day'] == 7, ['Day', 'Temperature']])
```

訓練集中位數為 21.5；訓練集與驗證集的缺失筆數從 0、1 變成 0、0，第 7 天填入 21.5。

</details>

## 4. 衍生欄位與摘要

簡單公式可直接套用整欄；需要修改資料時，使用明確的欄位指派或 `.loc`。
不必為每筆資料另寫函式。摘要用來理解資料與檢查處理結果。

In [ ]:
clean['Temperature_F'] = clean['Temperature'] * 9 / 5 + 32
display(clean[['Day', 'Temperature', 'Temperature_F']].head())
display(clean.groupby('Weather')['Temperature'].agg(['count', 'mean']))

## 5. 準備 X 與 y

`X` 是用來預測的特徵，`y` 是答案。兩者必須對應同一批樣本、相同順序。
假設任務是用氣溫與濕度預測是否下雨，答案 `Rain` 不可同時放進 `X`。
類別欄位的編碼與訓練／驗證切分在 scikit-learn 篇處理。

In [ ]:
X = clean[['Temperature', 'Humidity']]
y = clean['Rain']
print("X:", X.shape, "y:", y.shape)
print("轉成 NumPy 後：", X.to_numpy().shape)

### 根據缺失值選取樣本

原教材介紹 `notna()` 找出有資料的位置，以及用條件篩選資料。
多個條件可用 `&` 組合；篩選後再從同一張表選出特徵與答案，便能保留對應的索引。
`index` 可用來檢查樣本順序；分組的 `count` 會統計該欄非缺失的筆數。

In [ ]:
has_temperature = df['Temperature'].notna()
has_humidity = df['Humidity'].notna()
example_rows = df.loc[has_temperature & has_humidity]
print('篩選後索引：', example_rows.index)
print(example_rows.groupby('Rain')['Temperature'].agg(['count']))

### 小練習 2：準備對齊的特徵與答案

使用原始 `df` 的 `Temperature`、`Humidity` 預測 `Rain`。
先移除這三個欄位中任一欄有缺失值的樣本，再建立特徵表與答案。
印出兩者的 shape 和索引，以及保留下來的各類別樣本數；原始 `df` 保持不變。

<details>
<summary>參考答案</summary>

```python
valid = (df['Temperature'].notna() & df['Humidity'].notna()
         & df['Rain'].notna())
usable = df.loc[valid].copy()
ex_X = usable[['Temperature', 'Humidity']]
ex_y = usable['Rain']
print('shape:', ex_X.shape, ex_y.shape)
print('特徵索引：', ex_X.index)
print('答案索引：', ex_y.index)
print(usable.groupby('Rain')['Temperature'].agg(['count']))
```

`ex_X` 為 `(11, 2)`，`ex_y` 為 `(11,)`，兩者索引順序一致；沒下雨 9 筆、下雨 2 筆。

</details>

## 練習：下需求，也驗收結果

請 AI 用原始 `df` 找出高於平均氣溫的日期與總天數，說明缺失值如何處理。
再檢查：結果是否全都高於平均？總天數是否等於篩選後的列數？

In [ ]:
# 參考解答：mean 忽略缺失值；缺失氣溫不會符合大於條件。
mean_temp = df['Temperature'].mean()
above_mean = df.loc[df['Temperature'] > mean_temp, ['Day', 'Temperature']]
display(above_mean)
print("平均氣溫：", mean_temp, "高於平均的天數：", len(above_mean))